# 🫀Heart Disease Prediction

## Data Understanding
this notebook focuse on understanding the dataset before any preprocessing or modeling

### Objectives

* load the dataset
* Inspect its structure
* Understanding each feature
* Identify missing value
* Check duplicated records
* Generate desctiptive statistics
* Summarize initial observations

Author:Ali reza

In [1]:
# Importing the tools we need
from pathlib import Path
import pandas as pd
import numpy as np

In [13]:
# Project Path

PROJECT_ROOT = Path.cwd().parent
DATA_FILE = PROJECT_ROOT / "data" / "raw" / "heart.csv"

if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Dataset not found. Expected file at: {DATA_FILE.resolve()}"
    )

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Data file: {DATA_FILE.resolve()}")


Project root: G:\Heart-Disease-Prediction
Data file: G:\Heart-Disease-Prediction\data\raw\heart.csv


In [14]:
# Load Dataset

df = pd.read_csv(DATA_FILE)

print(f"Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")

df.head()

Dataset loaded successfully.
Dataset shape: (918, 12)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 97.6 KB


In [5]:
df.dtypes.to_frame(name="Data Type")

,Data Type
Age,int64
Sex,str
ChestPainType,str
RestingBP,int64
Cholesterol,int64
FastingBS,int64
RestingECG,str
MaxHR,int64
ExerciseAngina,str
Oldpeak,float64


In [16]:
df.isna().sum()

Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64

In [26]:
# Check categorical columns for unexpected values
categorical_cols = df.select_dtypes(include=['object', 'category', 'str']).columns

for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")

Sex: <ArrowStringArray>
['M', 'F']
Length: 2, dtype: str
ChestPainType: <ArrowStringArray>
['ATA', 'NAP', 'ASY', 'TA']
Length: 4, dtype: str
RestingECG: <ArrowStringArray>
['Normal', 'ST', 'LVH']
Length: 3, dtype: str
ExerciseAngina: <ArrowStringArray>
['N', 'Y']
Length: 2, dtype: str
ST_Slope: <ArrowStringArray>
['Up', 'Flat', 'Down']
Length: 3, dtype: str


### Initial Observations

- No missing values (`NaN`) were found in the dataset.
- The `Cholesterol` feature contains 172 records with a value of 0.
- The `RestingBP` feature contains 1 record with a value of 0.
- These values are medically questionable and will be investigated during the preprocessing stage.

In [9]:
# Check for duplicate rows
df.duplicated().sum()

np.int64(0)

In [10]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [22]:
# check zero & negative
print("Cholesterol == 0:", (df["Cholesterol"] == 0).sum())
print("RestingBP == 0:", (df["RestingBP"] == 0).sum())
print("Oldpeak < 0:", (df["Oldpeak"] < 0).sum())

Cholesterol == 0: 172
RestingBP == 0: 1
Oldpeak < 0: 13


In [12]:
# Examining the distribution of the target variable
df["HeartDisease"].value_counts()

HeartDisease
1    508
0    410
Name: count, dtype: int64

* Heart Disease (1): about 55.3%
* No Heart Disease (0): about 44.7%

In [41]:
# Schema and Target Validation
expected_columns = {
    "Age", "Sex", "ChestPainType", "RestingBP", "Cholesterol",
    "FastingBS", "RestingECG", "MaxHR", "ExerciseAngina",
    "Oldpeak", "ST_Slope", "HeartDisease"
}

assert set(df.columns) == expected_columns, "Schema mismatch."
assert df.columns.is_unique, "Duplicate column names detected."
assert df["HeartDisease"].isin([0, 1]).all(), "Target is not binary."

print("Schema validation passed.")


Schema validation passed.


In [46]:
# summary
data_quality_summary = {
    "rows": df.shape[0],
    "columns": df.shape[1],
    "missing_values": int(df.isna().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "cholesterol_zero": int((df["Cholesterol"] == 0).sum()),
    "restingbp_zero": int((df["RestingBP"] == 0).sum()),
    "oldpeak_negative": int((df["Oldpeak"] < 0).sum()),
}

pd.Series(data_quality_summary)


rows                918
columns              12
missing_values        0
duplicate_rows        0
cholesterol_zero    172
restingbp_zero        1
oldpeak_negative     13
dtype: int64

## Summary

- The dataset contains 918 observations and 12 features.
- No missing values (`NaN`) were detected.
- No duplicate records were found.
- `Cholesterol` contains 172 zero values, which are medically suspicious.
- `RestingBP` contains one zero value.
- The target variable is reasonably balanced (55.3% positive vs. 44.7% negative).
- Further investigation of suspicious values will be performed during the preprocessing stage.